# 01 Hypothesis validation

## Adrian Vazquez 

--- 

# 01_hypothesis_validation

### <b> Adrián Vazquez </b>

---

1. <b> Hipótesis 1: Leader realmente lidera: </b>

Para cada par: 

$return_{leader}(t)  →  return_{follower}(t+1, t+2, t+3...) $ 

Pregunta: Cuando el leader se mueve, ¿el follower se mueve después?

2. <b> Hipótesis 2: La correlación 60 días sirve </b>

Compara: Top correlation pairs vs random same-sector pairs

Pregunta: ¿Los pares top 50 tienen mejor relación lead-lag que pares aleatorios?

3. <b> Hipótesis 3: Market cap como leader tiene sentido.  </b>

Compara dos versiones: 

  -  Leader = mayor market cap
  - Leader = ticker que estadísticamente lidera más

Pregunta: ¿Market cap predice liderazgo o sólo es una regla intuitiva?

4. <b> Hipótesis 4: Los edge factors tienen sentido. </b>

aun no optimizar. Solo medir frecuencia

Pregunta:
 - ¿Cuántas señales genera?
 - ¿Son demasiado raras?
 - ¿O demasiado frecuentes?

5. <b> Hipótesis 5: La señal tiene drift posterior.</b>

Después de señal: follower forward return 5m, 15m, 30m, 60m

Pregunta: 
 - Después de la señal, ¿el follower sube más que en condiciones normales? <- Esta es la prueba más importante antes del backtest

 <b> Output que hay que buscar </b>

 | Hipótesis        |                        Métrica | Resultado          | Decisión              |
| ---------------- | -----------------------------: | ------------------ | --------------------- |
| Leader lidera    |                  Corr lead-lag | positiva / nula    | seguir / ajustar      |
| Top corr aporta  |           diferencia vs random | positiva / nula    | mantener / cambiar    |
| Market cap sirve | hit rate vs leader estadístico | mejor / peor       | mantener / reemplazar |
| Edge factors     |                señales por día | razonable / escaso | ajustar               |
| Drift posterior  |                 forward return | positivo / nulo    | pasar a backtest      |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import polars as pl
import pyarrow
import gc
from itertools import combinations
sys.path.append(str(Path("..")))
from src.ingest.open_parquet import abrir_parquet_data_pandas 

In [2]:
df_candidate_pairs = pd.read_csv('..\data\candidate_pairs.csv') 

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\avazq\AppData\Local\Temp\ipykernel_15336\3771880710.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_candidate_pairs = pd.read_csv('..\data\candidate_pairs.csv')


In [3]:
df_candidate_pairs

,stock_a,stock_b,sector
0,CCEP,KDP,BEVERAGES
1,CCEP,PEP,BEVERAGES
2,KDP,PEP,BEVERAGES
3,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)"
4,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES
...,...,...,...
185,SHOP,TEAM,SERVICES-PREPACKAGED SOFTWARE
186,SHOP,TTWO,SERVICES-PREPACKAGED SOFTWARE
187,SNPS,TEAM,SERVICES-PREPACKAGED SOFTWARE
188,SNPS,TTWO,SERVICES-PREPACKAGED SOFTWARE


## Research Decision: Correlation Definition

For pair ranking we define:

- Observation frequency: 1-minute
- Variable: log returns
- Correlation: Pearson correlation
- Window: 60 trading days (23,460 observations)

Lead-lag relationships will be evaluated separately in Hypothesis 1 following Lo & MacKinlay (1990).

The correct workflow would be:

 1. Read each cleared stock by symbol

 2. For each symbol: 
     - calculate log_return_1m = log(close_t / close_t-1)

 3. For each pair in candidate_pairs:
     - merge stock_a and stock_b by date

4. Calculate the 60-day rolling correlation
     - window = 60 × 391 = 23,460 observations

5. For each day:
     - obtain the most recent available rolling correlation

6. Rank all pairs by correlation

7. Select the Top 50

In [4]:
DATA_DIR = Path("../data/rth_1min_by_symbol")
OUT_DIR = Path("../data/rth_1min_log_returns_by_symbol")
OUT_DIR.mkdir(exist_ok=True)

def add_log_returns(symbol_file):
    df = pd.read_parquet(symbol_file)
    df = df.sort_values("date").copy()
    df["log_return_1m"] = np.log(df["close"] / df["close"].shift(1))
    return df

In [ ]:
# save parquets with log returns
for file in DATA_DIR.glob("*_rth_1min.parquet"):
    symbol = file.name.replace("_rth_1min.parquet", "")

    df_ret = add_log_returns(file)

    df_ret.to_parquet(OUT_DIR / f"{symbol}_rth_1min_returns.parquet",index=False)

In [5]:
sample = pd.read_parquet(OUT_DIR / "NVDA_rth_1min_returns.parquet")

sample[["date", "symbol", "close", "log_return_1m"]].head()

,date,symbol,close,log_return_1m
0,2021-06-01 09:30:00,NVDA,16.251499,NaN
1,2021-06-01 09:31:00,NVDA,16.256500,0.000308
2,2021-06-01 09:32:00,NVDA,16.221300,-0.002168
3,2021-06-01 09:33:00,NVDA,16.180799,-0.002500
4,2021-06-01 09:34:00,NVDA,16.153601,-0.001682


Pair Selection Methodology

For each trading day t:

1. Use the previous 60 trading days of 1-minute log returns.
2. Compute Pearson correlation for every candidate pair.
3. Rank all candidate pairs by correlation.
4. Select the Top 50 pairs.
5. Use this Top 50 universe for signal generation during day t.

In [ ]:
RETURNS_DIR = Path("../data/rth_1min_log_returns_by_symbol")
WINDOW_DAYS = 60
BARS_PER_DAY = 391
WINDOW_SIZE = WINDOW_DAYS * BARS_PER_DAY


def load_pair_returns(stock_a, stock_b, returns_dir=RETURNS_DIR):
    df_a = pd.read_parquet(returns_dir / f"{stock_a}_rth_1min_returns.parquet")
    df_b = pd.read_parquet(returns_dir / f"{stock_b}_rth_1min_returns.parquet")

    df_pair = (
        df_a[["date", "log_return_1m"]]
        .rename(columns={"log_return_1m": f"ret_{stock_a}"})
        .merge(
            df_b[["date", "log_return_1m"]].rename(columns={"log_return_1m": f"ret_{stock_b}"}),
            on="date",
            how="inner"
        )
        .sort_values("date")
        .dropna()
        .reset_index(drop=True)
    )

    df_pair["trading_day"] = df_pair["date"].dt.date

    return df_pair

def compute_daily_rolling_corr_for_pair(stock_a, stock_b):
    df_pair = load_pair_returns(stock_a, stock_b)

    corr_col = "rolling_corr_60d"

    df_pair[corr_col] = (
        df_pair[f"ret_{stock_a}"]
        .rolling(WINDOW_SIZE)
        .corr(df_pair[f"ret_{stock_b}"])
    )

    daily_corr = (
        df_pair
        .groupby("trading_day", as_index=False)
        .tail(1)
        [["trading_day", corr_col]]
        .copy()
    )

    daily_corr["stock_a"] = stock_a
    daily_corr["stock_b"] = stock_b

    return daily_corr[["trading_day", "stock_a", "stock_b", corr_col]]


test_corr = compute_daily_rolling_corr_for_pair("NVDA", "AMD")

test_corr.head(70)

,trading_day,stock_a,stock_b,rolling_corr_60d
389,2021-06-01,NVDA,AMD,NaN
780,2021-06-02,NVDA,AMD,NaN
1171,2021-06-03,NVDA,AMD,NaN
1562,2021-06-04,NVDA,AMD,NaN
1953,2021-06-07,NVDA,AMD,NaN
...,...,...,...,...
25804,2021-09-01,NVDA,AMD,0.539634
26195,2021-09-02,NVDA,AMD,0.538799
26586,2021-09-03,NVDA,AMD,0.537889
26977,2021-09-07,NVDA,AMD,0.537677


In [10]:
test_corr["rolling_corr_60d"].isna().sum(), len(test_corr)

(np.int64(60), 1176)

## run for 160 candidate pears and build daily_pair_correlation

In [11]:
def compute_daily_rolling_corr_for_all_pairs(candidate_pairs):
    results = []

    for i, row in candidate_pairs.iterrows():
        stock_a = row["stock_a"]
        stock_b = row["stock_b"]

        print(f"{i+1}/{len(candidate_pairs)} - {stock_a}-{stock_b}")

        pair_corr = compute_daily_rolling_corr_for_pair(stock_a, stock_b)
        pair_corr["sector"] = row["sector"]

        results.append(pair_corr)

    return pd.concat(results, ignore_index=True)

In [12]:
daily_pair_correlations = compute_daily_rolling_corr_for_all_pairs(df_candidate_pairs)

1/190 - CCEP-KDP
2/190 - CCEP-PEP
3/190 - KDP-PEP
4/190 - AMGN-GILD
5/190 - CHTR-CMCSA
6/190 - FTNT-PANW
7/190 - STX-WDC
8/190 - EXC-XEL
9/190 - PCAR-TSLA
10/190 - ALNY-INSM
11/190 - ALNY-REGN
12/190 - ALNY-VRTX
13/190 - INSM-REGN
14/190 - INSM-VRTX
15/190 - REGN-VRTX
16/190 - AMZN-PDD
17/190 - COST-WMT
18/190 - ADI-AMAT
19/190 - ADI-AMD
20/190 - ADI-ASML
21/190 - ADI-AVGO
22/190 - ADI-INTC
23/190 - ADI-MCHP
24/190 - ADI-MPWR
25/190 - ADI-MRVL
26/190 - ADI-MU
27/190 - ADI-NVDA
28/190 - ADI-NXPI
29/190 - ADI-TXN
30/190 - AMAT-AMD
31/190 - AMAT-ASML
32/190 - AMAT-AVGO
33/190 - AMAT-INTC
34/190 - AMAT-MCHP
35/190 - AMAT-MPWR
36/190 - AMAT-MRVL
37/190 - AMAT-MU
38/190 - AMAT-NVDA
39/190 - AMAT-NXPI
40/190 - AMAT-TXN
41/190 - AMD-ASML
42/190 - AMD-AVGO
43/190 - AMD-INTC
44/190 - AMD-MCHP
45/190 - AMD-MPWR
46/190 - AMD-MRVL
47/190 - AMD-MU
48/190 - AMD-NVDA
49/190 - AMD-NXPI
50/190 - AMD-TXN
51/190 - ASML-AVGO
52/190 - ASML-INTC
53/190 - ASML-MCHP
54/190 - ASML-MPWR
55/190 - ASML-MRVL
56/190

In [20]:
daily_pair_correlations.head(62)

,trading_day,stock_a,stock_b,rolling_corr_60d,sector
0,2021-12-31,CCEP,KDP,NaN,BEVERAGES
1,2022-01-03,CCEP,KDP,NaN,BEVERAGES
2,2022-01-04,CCEP,KDP,NaN,BEVERAGES
3,2022-01-05,CCEP,KDP,NaN,BEVERAGES
4,2022-01-06,CCEP,KDP,NaN,BEVERAGES
...,...,...,...,...,...
57,2022-03-24,CCEP,KDP,NaN,BEVERAGES
58,2022-03-25,CCEP,KDP,NaN,BEVERAGES
59,2022-03-28,CCEP,KDP,NaN,BEVERAGES
60,2022-03-29,CCEP,KDP,0.408533,BEVERAGES


In [14]:
daily_pair_correlations.shape

(169641, 5)

In [15]:
daily_pair_correlations["rolling_corr_60d"].isna().sum()

np.int64(10210)

In [18]:
daily_pair_correlations.to_parquet("../data/daily_pair_correlations_60d.parquet",index=False)

## Hypothesis 1 — Does the Leader Actually Lead?

### Objective

The strategy assumes that, within each selected pair, the larger market-cap stock acts as the **leader** and the smaller market-cap stock acts as the **follower**.

This hypothesis tests whether leader returns contain information about future follower returns.

### Hypothesis

For each Top-50 pair selected by rolling 60-day correlation:

$ r_{leader,t} \rightarrow r_{follower,t+k} $

where \(k\) represents future lags such as 1, 5, 15 and 30 minutes.

### Test

For each pair and trading day:

1. Define leader as the stock with the larger market capitalization.
2. Define follower as the other stock.
3. Compute lagged correlations between:
   - leader return at time $ t $
   - follower return at time $ t+k $

### Decision Rule

Evidence supports the hypothesis if:

- lead-lag correlations are positive on average, and
- they are stronger than the reverse direction.

In [21]:
RETURNS_DIR = Path("../data/rth_1min_log_returns_by_symbol")
UNIVERSE_PATH = Path("../data/universe_metadata.csv")
CORR_PATH = Path("../data/daily_pair_correlations_60d.parquet")

In [ ]:
# First we built the daily Top 50
daily_pair_correlations = pd.read_parquet(CORR_PATH)

daily_top50_pairs = (daily_pair_correlations.dropna(subset=["rolling_corr_60d"]).sort_values(["trading_day", "rolling_corr_60d"], ascending=[True, False])
    .groupby("trading_day", as_index=False)
    .head(50)
    .reset_index(drop=True)
)

daily_top50_pairs.head()
daily_top50_pairs.to_csv("../data/daily_top50_pairs.csv", index=False)

,trading_day,stock_a,stock_b,rolling_corr_60d,sector
0,2021-08-25,GOOG,GOOGL,0.714106,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING..."
1,2021-08-25,ADI,TXN,0.677198,SEMICONDUCTORS & RELATED DEVICES
2,2021-08-25,MCHP,NXPI,0.675279,SEMICONDUCTORS & RELATED DEVICES
3,2021-08-25,ADI,MCHP,0.656578,SEMICONDUCTORS & RELATED DEVICES
4,2021-08-25,MCHP,TXN,0.652763,SEMICONDUCTORS & RELATED DEVICES


In [ ]:
# we added leader/follower 
universe_df = pd.read_csv(UNIVERSE_PATH)

market_cap_map = universe_df.set_index("symbol")["market_cap"].to_dict()

def assign_leader_follower(row):
    a = row["stock_a"]
    b = row["stock_b"]

    if market_cap_map[a] >= market_cap_map[b]:
        return pd.Series({
            "leader": a,
            "follower": b,
            "leader_market_cap": market_cap_map[a],
            "follower_market_cap": market_cap_map[b]
        })
    else:
        return pd.Series({
            "leader": b,
            "follower": a,
            "leader_market_cap": market_cap_map[b],
            "follower_market_cap": market_cap_map[a]
        })

leader_follower_cols = daily_top50_pairs.apply(assign_leader_follower, axis=1)

daily_top50_pairs = pd.concat(
    [daily_top50_pairs, leader_follower_cols],
    axis=1
)

daily_top50_pairs.head()

,trading_day,stock_a,stock_b,rolling_corr_60d,sector,leader,follower,leader_market_cap,follower_market_cap
0,2021-08-25,GOOG,GOOGL,0.714106,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING...",GOOG,GOOGL,4.022414e+12,4.018794e+12
1,2021-08-25,ADI,TXN,0.677198,SEMICONDUCTORS & RELATED DEVICES,TXN,ADI,2.021884e+11,1.565761e+11
2,2021-08-25,MCHP,NXPI,0.675279,SEMICONDUCTORS & RELATED DEVICES,NXPI,MCHP,5.709487e+10,4.227916e+10
3,2021-08-25,ADI,MCHP,0.656578,SEMICONDUCTORS & RELATED DEVICES,ADI,MCHP,1.565761e+11,4.227916e+10
4,2021-08-25,MCHP,TXN,0.652763,SEMICONDUCTORS & RELATED DEVICES,TXN,MCHP,2.021884e+11,4.227916e+10


In [24]:
# function to measure leader/ follower in a pair 
def compute_pair_lead_lag(leader, follower, lags=(1, 5, 15, 30)):
    df_leader = pd.read_parquet(RETURNS_DIR / f"{leader}_rth_1min_returns.parquet")
    df_follower = pd.read_parquet( RETURNS_DIR / f"{follower}_rth_1min_returns.parquet")

    df_pair = (df_leader[["date", "log_return_1m"]].rename(columns={"log_return_1m": "leader_ret"}).merge(df_follower[["date", "log_return_1m"]]
            .rename(columns={"log_return_1m": "follower_ret"}),on="date",how="inner").dropna().sort_values("date").reset_index(drop=True))

    results = {}

    for lag in lags:
        results[f"leader_to_follower_lag_{lag}m"] = (df_pair["leader_ret"].corr(df_pair["follower_ret"].shift(-lag)))

        results[f"follower_to_leader_lag_{lag}m"] = (df_pair["follower_ret"].corr(df_pair["leader_ret"].shift(-lag)))

    return results

In [25]:
unique_top_pairs = (daily_top50_pairs[["leader", "follower"]].drop_duplicates().reset_index(drop=True))

lead_lag_results = []

for i, row in unique_top_pairs.iterrows():
    leader = row["leader"]
    follower = row["follower"]

    print(f"{i+1}/{len(unique_top_pairs)} - {leader}->{follower}")

    metrics = compute_pair_lead_lag(leader, follower)

    lead_lag_results.append({"leader": leader,"follower": follower,**metrics})

lead_lag_df = pd.DataFrame(lead_lag_results)

lead_lag_df.head()

1/118 - GOOG->GOOGL
2/118 - TXN->ADI
3/118 - NXPI->MCHP
4/118 - ADI->MCHP
5/118 - TXN->MCHP
6/118 - XEL->EXC
7/118 - TXN->NXPI
8/118 - ADI->NXPI
9/118 - AMAT->ADI
10/118 - MU->AMAT
11/118 - AMAT->NXPI
12/118 - AMAT->MCHP
13/118 - AMAT->TXN
14/118 - AVGO->TXN
15/118 - AVGO->ADI
16/118 - MSFT->ADBE
17/118 - ADBE->CDNS
18/118 - NVDA->AMD
19/118 - ADI->MRVL
20/118 - SNPS->CDNS
21/118 - MRVL->MCHP
22/118 - AMAT->MRVL
23/118 - TXN->MRVL
24/118 - AVGO->MCHP
25/118 - MU->ADI
26/118 - AMGN->GILD
27/118 - MRVL->NXPI
28/118 - AVGO->AMAT
29/118 - MU->NXPI
30/118 - AVGO->NXPI
31/118 - CDNS->ADSK
32/118 - MU->MCHP
33/118 - MU->TXN
34/118 - ASML->AMAT
35/118 - MSFT->CDNS
36/118 - AVGO->MRVL
37/118 - ASML->NXPI
38/118 - ADBE->SNPS
39/118 - ADBE->ADSK
40/118 - MU->MRVL
41/118 - ASML->MCHP
42/118 - INTU->ADBE
43/118 - INTU->CDNS
44/118 - ASML->ADI
45/118 - MSFT->INTU
46/118 - AVGO->MU
47/118 - MSFT->ADSK
48/118 - INTC->TXN
49/118 - AVGO->ASML
50/118 - SNPS->ADSK
51/118 - NVDA->MRVL
52/118 - AMD->MRVL
53

,leader,follower,leader_to_follower_lag_1m,follower_to_leader_lag_1m,leader_to_follower_lag_5m,follower_to_leader_lag_5m,leader_to_follower_lag_15m,follower_to_leader_lag_15m,leader_to_follower_lag_30m,follower_to_leader_lag_30m
0,GOOG,GOOGL,-0.002154,0.002210,-0.001580,-0.002835,0.004973,0.002925,-0.000945,-0.001186
1,TXN,ADI,0.018909,0.006584,-0.001070,-0.005206,0.003961,0.003309,0.000441,-0.000253
2,NXPI,MCHP,0.026492,0.065470,-0.002995,0.001926,0.006047,0.005297,0.003722,-0.001147
3,ADI,MCHP,0.007694,0.023130,-0.003904,0.000891,0.008018,0.003521,0.002485,-0.000271
4,TXN,MCHP,0.012853,0.019628,-0.005619,0.001461,0.005759,0.001548,0.001218,-0.001207


In [26]:
lead_lag_df.describe()

,leader_to_follower_lag_1m,follower_to_leader_lag_1m,leader_to_follower_lag_5m,follower_to_leader_lag_5m,leader_to_follower_lag_15m,follower_to_leader_lag_15m,leader_to_follower_lag_30m,follower_to_leader_lag_30m
count,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000
mean,0.013757,0.011282,-0.002295,-0.000671,0.003171,0.002743,-0.000047,-0.000056
std,0.014552,0.016358,0.003764,0.002535,0.002532,0.002396,0.002188,0.001857
min,-0.009427,-0.008809,-0.009524,-0.006223,-0.003917,-0.002325,-0.005615,-0.005032
25%,0.003111,-0.001059,-0.004859,-0.002238,0.001643,0.001052,-0.001230,-0.001142
50%,0.009431,0.004788,-0.002598,-0.000428,0.003019,0.002665,0.000097,0.000075
75%,0.024002,0.019490,0.000039,0.001193,0.004752,0.004596,0.001383,0.001286
max,0.058169,0.065470,0.008571,0.005650,0.009813,0.008546,0.004808,0.004162


In [27]:
lead_lag_df[
    [
        "leader_to_follower_lag_1m",
        "follower_to_leader_lag_1m"
    ]
].mean()

leader_to_follower_lag_1m    0.013757
follower_to_leader_lag_1m    0.011282
dtype: float64

In [28]:
(
    lead_lag_df["leader_to_follower_lag_1m"]
    >
    lead_lag_df["follower_to_leader_lag_1m"]
).mean()

np.float64(0.5677966101694916)

In [ ]:
lead_lag_df.to_parquet("../data/hypothesis_1_lead_lag_results.parquet", index=False)

### Preliminary Result

The market-cap rule shows weak evidence of directional leadership.

Results:

- Average leader→follower lag-1 correlation: 0.0138
- Average follower→leader lag-1 correlation: 0.0113
- Leader outperformed follower in 56.8% of pairs

Interpretation:

The evidence is consistent with a mild lead-lag effect in the expected direction, although the magnitude appears weak. Market capitalization may capture part of the leadership structure, but likely does not fully explain information transmission across pairs.

## Hypothesis 2 — Does Rolling Correlation Add Value?

### Objective

The strategy ranks candidate pairs using a 60-day rolling correlation and trades only the Top 50 pairs.

This hypothesis evaluates whether the rolling-correlation ranking identifies pairs with stronger lead-lag relationships than randomly selected same-sector pairs.

### Null Hypothesis

Top correlation pairs do not exhibit stronger lead-lag effects than random same-sector pairs.

### Alternative Hypothesis

Top correlation pairs exhibit stronger lead-lag effects than random same-sector pairs.

### Evaluation Metric

Lag-1 lead-lag correlation:

$ corr(
r_leader(t),
r_follower(t+1)
) $

### Decision Rule

If Top50 pairs show higher average lead-lag correlation than random same-sector pairs, the rolling-correlation filter adds value.

In [ ]:
# Random benchmark:
# take randomly 50 pairs
# mesure lead-lag average 
#  repeat 1,000 times 
# If the Top50 falls within the 80th, 90th, and 95th percentiles of the random distribution, then rolling correlation does contribute.

observed_top50_mean = (lead_lag_df["leader_to_follower_lag_1m"].mean())

# When the leader moves in one minute, the follower tends to move slightly in the same direction one minute later.

np.float64(0.013757028668868656)

In [ ]:
n_top_pairs = len(lead_lag_df)
n_top_pairs

# what does it mean ? 118 that appeared at least once in a daily Top 50 list
# candidate_pairs = 190, But the Top 50 ranking changed day by day.
# 118 pairs were considered "interesting" at some point by the strategy.


118

In [ ]:
# QUESTION :  is 0.013757 special? or Does any random selection of pairs produce something similar?

# First we need to calculate the same lead-lag for all 190 candidate pairs
candidate_pairs_with_roles = df_candidate_pairs.copy()
roles = candidate_pairs_with_roles.apply(assign_leader_follower,axis=1)
candidate_pairs_with_roles = pd.concat([candidate_pairs_with_roles, roles],axis=1)
candidate_pairs_with_roles.head()
candidate_pairs_with_roles.shape

(190, 7)

In [35]:
# Now we're going to build the base dataset for the benchmark
candidate_leadlag_results = []

for i, row in candidate_pairs_with_roles.iterrows():

    leader = row["leader"]
    follower = row["follower"]

    print(f"{i+1}/{len(candidate_pairs_with_roles)} " f"{leader}->{follower}")

    metrics = compute_pair_lead_lag(leader,follower)

    candidate_leadlag_results.append({"leader": leader,"follower": follower,"sector": row["sector"],**metrics})

candidate_leadlag_df = pd.DataFrame(candidate_leadlag_results)
candidate_leadlag_df.shape

1/190 CCEP->KDP
2/190 PEP->CCEP
3/190 PEP->KDP
4/190 AMGN->GILD
5/190 CMCSA->CHTR
6/190 PANW->FTNT
7/190 WDC->STX
8/190 XEL->EXC
9/190 TSLA->PCAR
10/190 ALNY->INSM
11/190 REGN->ALNY
12/190 VRTX->ALNY
13/190 REGN->INSM
14/190 VRTX->INSM
15/190 VRTX->REGN
16/190 AMZN->PDD
17/190 WMT->COST
18/190 AMAT->ADI
19/190 AMD->ADI
20/190 ASML->ADI
21/190 AVGO->ADI
22/190 INTC->ADI
23/190 ADI->MCHP
24/190 ADI->MPWR
25/190 ADI->MRVL
26/190 MU->ADI
27/190 NVDA->ADI
28/190 ADI->NXPI
29/190 TXN->ADI
30/190 AMD->AMAT
31/190 ASML->AMAT
32/190 AVGO->AMAT
33/190 INTC->AMAT
34/190 AMAT->MCHP
35/190 AMAT->MPWR
36/190 AMAT->MRVL
37/190 MU->AMAT
38/190 NVDA->AMAT
39/190 AMAT->NXPI
40/190 AMAT->TXN
41/190 ASML->AMD
42/190 AVGO->AMD
43/190 AMD->INTC
44/190 AMD->MCHP
45/190 AMD->MPWR
46/190 AMD->MRVL
47/190 MU->AMD
48/190 NVDA->AMD
49/190 AMD->NXPI
50/190 AMD->TXN
51/190 AVGO->ASML
52/190 ASML->INTC
53/190 ASML->MCHP
54/190 ASML->MPWR
55/190 ASML->MRVL
56/190 ASML->MU
57/190 NVDA->ASML
58/190 ASML->NXPI
59/190 AS

(190, 11)

In [37]:
# we already have the correct dataset to perform the benchmark.
candidate_leadlag_df.head(2)

,leader,follower,sector,leader_to_follower_lag_1m,follower_to_leader_lag_1m,leader_to_follower_lag_5m,follower_to_leader_lag_5m,leader_to_follower_lag_15m,follower_to_leader_lag_15m,leader_to_follower_lag_30m,follower_to_leader_lag_30m
0,CCEP,KDP,BEVERAGES,0.004110,0.026390,-0.002551,0.003729,0.003299,0.002008,0.001963,0.002590
1,PEP,CCEP,BEVERAGES,0.030483,0.010354,0.004506,-0.002036,-0.000826,0.005631,0.001255,0.001996


In [39]:
# Bootstrap
bootstrap_means = []

for _ in range(10000):

    sample = candidate_leadlag_df.sample(n=n_top_pairs,replace=False)

    bootstrap_means.append(sample["leader_to_follower_lag_1m"].mean())

bootstrap_means = np.array(bootstrap_means)
bootstrap_means.mean()

np.float64(0.01432898852757192)

In [40]:
bootstrap_means.std()

np.float64(0.0016228793520580618)

In [ ]:
np.percentile(bootstrap_means,[5, 25, 50, 75, 95])


array([0.01168811, 0.01318236, 0.01430491, 0.01547147, 0.01706619])

In [42]:
(bootstrap_means < observed_top50_mean).mean()

np.float64(0.3654)

 Do pairs selected using 60-day rolling correlation have a greater lead-lag effect than randomly chosen pairs? 
- We found no evidence that it did.

Do Top50 pairs selected by rolling 60-day correlation exhibit stronger lead-lag effects than randomly selected same-sector pairs?

<b> Result: </b> No meaningful improvement was observed.

The average leader-to-follower lag correlation of the Top50 universe (0.0138) fell near the center of the bootstrap distribution generated from random candidate pairs.

Conclusion

Rolling correlation appears useful for identifying co-moving assets, but does not appear to identify stronger lead-lag relationships by itself.

# ¿Market cap realmente sirve para definir quién es el leader?
La estrategia define:
 - Leader = mayor market cap
 - Follower = menor market cap
 Pero ahora vamos a comparar esa regla contra una alternativa estadística: leader estadístico = dirección con mayor lead-lag

Compararemos:

leader_to_follower_lag_1m vs follower_to_leader_lag_1m

Si: leader_to_follower_lag_1m > follower_to_leader_lag_1m

entonces market cap acertó.

Si no, el liderazgo estadístico va en dirección contraria.

In [49]:
candidate_leadlag_df["market_cap_leader_correct"] = (candidate_leadlag_df["leader_to_follower_lag_1m"] > candidate_leadlag_df["follower_to_leader_lag_1m"] )
market_cap_accuracy = candidate_leadlag_df["market_cap_leader_correct"].mean()
market_cap_accuracy

np.float64(0.5789473684210527)

In [50]:
candidate_leadlag_df[["leader_to_follower_lag_1m","follower_to_leader_lag_1m"]].mean()

leader_to_follower_lag_1m    0.014304
follower_to_leader_lag_1m    0.010314
dtype: float64

In [51]:
candidate_leadlag_df["lag_difference"] = (candidate_leadlag_df["leader_to_follower_lag_1m"] - candidate_leadlag_df["follower_to_leader_lag_1m"])
candidate_leadlag_df["lag_difference"].describe()

count    190.000000
mean       0.003991
std        0.034079
min       -0.225169
25%       -0.010587
50%        0.003766
75%        0.018430
max        0.199479
Name: lag_difference, dtype: float64

In [52]:
(candidate_leadlag_df["lag_difference"] > 0).mean()

np.float64(0.5789473684210527)

Does Market Capitalization Correctly Identify the Leader?

Does assigning the larger market-cap stock as the leader correctly identify the direction of the lead-lag relationship?

Methodology

For each candidate pair:

Assign leader and follower using market capitalization.

Compute:
- Leader → Follower lag-1 correlation
- Follower → Leader lag-1 correlation
- Define a correct assignment when:
  - Leader → Follower > Follower → Leader

| Metric                |   Value |
| --------------------- | ------: |
| Candidate Pairs       |     190 |
| Accuracy              |   57.9% |
| Mean Lag Difference   | 0.00399 |
| Median Lag Difference | 0.00377 |

Interpretation

The larger market-cap stock correctly identifies the statistically dominant direction of information flow in approximately 58% of candidate pairs.

The effect is positive but moderate, suggesting that market capitalization contains useful information regarding leadership, although it is far from a perfect predictor.

Conclusion

Market capitalization provides a meaningful but incomplete signal for leader identification.

--- 

## Hypothesis 4: Do the Edge Factors Generate a Reasonable Number of Trading Signals?

### Research Question

Do the proposed edge factors (0.6% for leaders and 0.2% for followers) generate a reasonable frequency of trading opportunities?

Before evaluating profitability, it is necessary to verify that the signal-generation mechanism produces a realistic number of trades.

If the thresholds are too restrictive, the strategy may rarely trade. If they are too permissive, the strategy may generate excessive noise and overtrade.

### Methodology

For each Top50 pair:

1. Compute the 15-minute SMA for both leader and follower.
2. Apply the edge factors defined by the strategy:

   * Leader Long SMA = SMA × (1 + 0.006)
   * Follower Long SMA = SMA × (1 + 0.002)
3. Count the number of times the entry conditions are satisfied.
4. Aggregate signal counts across all pairs and trading days.

### Objective

Measure signal frequency only.

No positions, PnL, transaction costs, or backtesting will be considered at this stage.

### Success Criteria

The edge factors should generate a reasonable number of signals:

* Not so rare that the strategy is effectively inactive.
* Not so frequent that signals become indistinguishable from noise.

The output of this hypothesis will be the empirical distribution of signal frequency across pairs and trading days.


In [53]:
#  the strategy say : 
# Leader Long SMA = SMA15 × (1 + 0.006)
# Follower Long SMA = SMA15 × (1 + 0.002)
# Generates a signal when: leader_price > leader_long_SMA  AND follower_price < follower_long_SMA

def load_pair_prices(leader, follower):
    df_leader = pd.read_parquet(RETURNS_DIR / f"{leader}_rth_1min_returns.parquet")
    df_follower = pd.read_parquet(RETURNS_DIR / f"{follower}_rth_1min_returns.parquet")
    df = (df_leader[["date", "close"]].rename(columns={"close": "leader_price"}).merge(df_follower[["date", "close"]].rename(columns={"close": "follower_price"}),
            on="date",how="inner").sort_values("date").reset_index(drop=True))
    return df

pair_df = load_pair_prices("NVDA","AMD")

pair_df["leader_sma15"] = (pair_df["leader_price"].rolling(15).mean())
pair_df["follower_sma15"] = (pair_df["follower_price"].rolling(15).mean())
pair_df["leader_long_sma"] = (pair_df["leader_sma15"] * 1.006)
pair_df["follower_long_sma"] = (pair_df["follower_sma15"] * 1.002)

pair_df[["date","leader_price","leader_sma15","leader_long_sma","follower_price","follower_sma15","follower_long_sma"]].head(20)

,date,leader_price,leader_sma15,leader_long_sma,follower_price,follower_sma15,follower_long_sma
0,2021-06-01 09:30:00,16.251499,NaN,NaN,80.919998,NaN,NaN
1,2021-06-01 09:31:00,16.256500,NaN,NaN,81.010002,NaN,NaN
2,2021-06-01 09:32:00,16.221300,NaN,NaN,81.239998,NaN,NaN
3,2021-06-01 09:33:00,16.180799,NaN,NaN,81.258598,NaN,NaN
4,2021-06-01 09:34:00,16.153601,NaN,NaN,81.500000,NaN,NaN
5,2021-06-01 09:35:00,16.180500,NaN,NaN,81.735001,NaN,NaN
6,2021-06-01 09:36:00,16.153799,NaN,NaN,81.932297,NaN,NaN
7,2021-06-01 09:37:00,16.133801,NaN,NaN,82.019997,NaN,NaN
8,2021-06-01 09:38:00,16.137600,NaN,NaN,81.950104,NaN,NaN
9,2021-06-01 09:39:00,16.193001,NaN,NaN,82.480003,NaN,NaN


In [59]:
# Long signal condition
pair_df["long_signal"] = ((pair_df["leader_price"] > pair_df["leader_long_sma"]) & (pair_df["follower_price"] < pair_df["follower_long_sma"]))

# Trading day for signal-start logic
pair_df["trading_day"] = pair_df["date"].dt.date

# Previous signal within the same trading day
prev_signal = (pair_df.groupby("trading_day")["long_signal"].shift(1).fillna(False).astype(bool))

# Signal start = transition from False to True
pair_df["long_signal_start"] = (pair_df["long_signal"].astype(bool) & (~prev_signal))

pair_df["long_signal"].sum(), pair_df["long_signal_start"].sum()

(np.int64(1591), np.int64(762))

In [60]:
pair_df.loc[pair_df["long_signal_start"],["date","leader_price","leader_long_sma","follower_price","follower_long_sma"]].head(20)

,date,leader_price,leader_long_sma,follower_price,follower_long_sma
312,2021-06-01 14:42:00,16.327999,16.321217,81.641197,81.713768
787,2021-06-03 09:35:00,16.897699,16.869788,81.750000,81.956393
1197,2021-06-04 09:54:00,17.450001,17.441981,81.205002,81.240343
1564,2021-06-07 09:30:00,17.707500,17.702850,81.264999,81.746273
1569,2021-06-07 09:35:00,17.758801,17.733439,81.529999,81.622259
4749,2021-06-17 10:27:00,18.560499,18.517589,83.120003,83.145747
5086,2021-06-18 09:33:00,18.981800,18.806178,84.279999,84.693175
5141,2021-06-18 10:28:00,19.312799,19.312155,84.980003,85.202658
6257,2021-06-23 09:31:00,19.058001,19.024660,83.589996,83.791528
7462,2021-06-28 10:03:00,19.898899,19.884368,87.509003,87.536343


In [61]:
# Short SMA thresholds
pair_df["leader_short_sma"] = pair_df["leader_sma15"] * (1 - 0.006)
pair_df["follower_short_sma"] = pair_df["follower_sma15"] * (1 - 0.002)

# Short signal condition
pair_df["short_signal"] = ((pair_df["leader_price"] < pair_df["leader_short_sma"]) & (pair_df["follower_price"] < pair_df["follower_short_sma"]))

# Previous short signal within the same trading day
prev_short_signal = (pair_df.groupby("trading_day")["short_signal"].shift(1).fillna(False).astype(bool))

# Short signal start = transition from False to True
pair_df["short_signal_start"] = (pair_df["short_signal"].astype(bool) & (~prev_short_signal))

pair_df["short_signal"].sum(), pair_df["short_signal_start"].sum()

(np.int64(10503), np.int64(3361))

In [62]:
pair_df.loc[pair_df["short_signal_start"],["date","leader_price","leader_short_sma","follower_price","follower_short_sma"]].head(20)

,date,leader_price,leader_short_sma,follower_price,follower_short_sma
44,2021-06-01 10:14:00,15.938500,15.959896,81.719902,81.968095
800,2021-06-03 09:48:00,16.653000,16.663926,80.779999,81.043043
1578,2021-06-07 09:44:00,17.556000,17.567499,81.160004,81.161745
1613,2021-06-07 10:19:00,17.458401,17.475302,81.114998,81.271144
1640,2021-06-07 10:46:00,17.408501,17.416457,81.254997,81.260493
1978,2021-06-08 09:53:00,17.394300,17.394907,81.080002,81.673659
2065,2021-06-08 11:20:00,17.286501,17.287032,80.599998,80.635746
3911,2021-06-15 09:31:00,17.837500,17.865440,81.029999,81.174612
5474,2021-06-21 09:30:00,18.499800,18.582393,84.102997,84.375443
5523,2021-06-21 10:19:00,17.902500,17.909455,82.518700,82.803308


In [63]:
def compute_signal_frequency_for_pair(leader, follower):
    df = load_pair_prices(leader, follower)

    df["trading_day"] = df["date"].dt.date

    df["leader_sma15"] = df["leader_price"].rolling(15).mean()
    df["follower_sma15"] = df["follower_price"].rolling(15).mean()

    df["leader_long_sma"] = df["leader_sma15"] * 1.006
    df["follower_long_sma"] = df["follower_sma15"] * 1.002

    df["leader_short_sma"] = df["leader_sma15"] * (1 - 0.006)
    df["follower_short_sma"] = df["follower_sma15"] * (1 - 0.002)

    df["long_signal"] = ((df["leader_price"] > df["leader_long_sma"]) & (df["follower_price"] < df["follower_long_sma"]))

    df["short_signal"] = ((df["leader_price"] < df["leader_short_sma"]) &(df["follower_price"] < df["follower_short_sma"]))

    prev_long = (df.groupby("trading_day")["long_signal"].shift(1).fillna(False).astype(bool))

    prev_short = (df.groupby("trading_day")["short_signal"].shift(1).fillna(False).astype(bool))
    df["long_signal_start"] = df["long_signal"].astype(bool) & (~prev_long)
    df["short_signal_start"] = df["short_signal"].astype(bool) & (~prev_short)

    n_days = df["trading_day"].nunique()

    return {
        "leader": leader,
        "follower": follower,
        "n_days": n_days,
        "long_minutes": df["long_signal"].sum(),
        "long_starts": df["long_signal_start"].sum(),
        "short_minutes": df["short_signal"].sum(),
        "short_starts": df["short_signal_start"].sum(),
        "long_starts_per_day": df["long_signal_start"].sum() / n_days,
        "short_starts_per_day": df["short_signal_start"].sum() / n_days,
    }

In [64]:
compute_signal_frequency_for_pair("NVDA", "AMD")

{'leader': 'NVDA',
 'follower': 'AMD',
 'n_days': 1176,
 'long_minutes': np.int64(1591),
 'long_starts': np.int64(762),
 'short_minutes': np.int64(10503),
 'short_starts': np.int64(3361),
 'long_starts_per_day': np.float64(0.6479591836734694),
 'short_starts_per_day': np.float64(2.8579931972789114)}

In [65]:
signal_frequency_results = []

for i, row in unique_top_pairs.iterrows():
    leader = row["leader"]
    follower = row["follower"]

    print(f"{i+1}/{len(unique_top_pairs)} - {leader}->{follower}")

    result = compute_signal_frequency_for_pair(leader, follower)
    signal_frequency_results.append(result)

signal_frequency_df = pd.DataFrame(signal_frequency_results)

signal_frequency_df.head()

1/118 - GOOG->GOOGL
2/118 - TXN->ADI
3/118 - NXPI->MCHP
4/118 - ADI->MCHP
5/118 - TXN->MCHP
6/118 - XEL->EXC
7/118 - TXN->NXPI
8/118 - ADI->NXPI
9/118 - AMAT->ADI
10/118 - MU->AMAT
11/118 - AMAT->NXPI
12/118 - AMAT->MCHP
13/118 - AMAT->TXN
14/118 - AVGO->TXN
15/118 - AVGO->ADI
16/118 - MSFT->ADBE
17/118 - ADBE->CDNS
18/118 - NVDA->AMD
19/118 - ADI->MRVL
20/118 - SNPS->CDNS
21/118 - MRVL->MCHP
22/118 - AMAT->MRVL
23/118 - TXN->MRVL
24/118 - AVGO->MCHP
25/118 - MU->ADI
26/118 - AMGN->GILD
27/118 - MRVL->NXPI
28/118 - AVGO->AMAT
29/118 - MU->NXPI
30/118 - AVGO->NXPI
31/118 - CDNS->ADSK
32/118 - MU->MCHP
33/118 - MU->TXN
34/118 - ASML->AMAT
35/118 - MSFT->CDNS
36/118 - AVGO->MRVL
37/118 - ASML->NXPI
38/118 - ADBE->SNPS
39/118 - ADBE->ADSK
40/118 - MU->MRVL
41/118 - ASML->MCHP
42/118 - INTU->ADBE
43/118 - INTU->CDNS
44/118 - ASML->ADI
45/118 - MSFT->INTU
46/118 - AVGO->MU
47/118 - MSFT->ADSK
48/118 - INTC->TXN
49/118 - AVGO->ASML
50/118 - SNPS->ADSK
51/118 - NVDA->MRVL
52/118 - AMD->MRVL
53

,leader,follower,n_days,long_minutes,long_starts,short_minutes,short_starts,long_starts_per_day,short_starts_per_day
0,GOOG,GOOGL,1176,1,1,3121,884,0.000850,0.751701
1,TXN,ADI,1176,190,109,2793,747,0.092687,0.635204
2,NXPI,MCHP,1176,539,284,5889,1713,0.241497,1.456633
3,ADI,MCHP,1176,306,151,3595,989,0.128401,0.840986
4,TXN,MCHP,1176,184,95,2773,753,0.080782,0.640306


In [66]:
signal_frequency_df.describe()

,n_days,long_minutes,long_starts,short_minutes,short_starts,long_starts_per_day,short_starts_per_day
count,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000
mean,1105.347458,1714.110169,722.135593,4689.474576,1469.593220,0.702470,1.367009
std,197.551092,1285.227991,528.720240,2505.253044,827.447246,0.592422,0.786696
min,275.000000,1.000000,1.000000,363.000000,101.000000,0.000850,0.221939
25%,1176.000000,664.250000,304.000000,3093.250000,905.500000,0.288655,0.820763
50%,1176.000000,1338.500000,537.000000,4108.500000,1198.000000,0.516592,1.147109
75%,1176.000000,2585.000000,1069.000000,6649.500000,2089.250000,1.034402,1.949830
max,1176.000000,5403.000000,2126.000000,10503.000000,3361.000000,2.996364,3.592727


In [67]:
signal_frequency_df[["long_starts_per_day", "short_starts_per_day"]].mean()

long_starts_per_day     0.702470
short_starts_per_day    1.367009
dtype: float64

In [68]:
signal_frequency_df.to_parquet("../data/hypothesis_4_signal_frequency.parquet",index=False)


- They are not overly restrictive.
- They are not overly permissive.
- They generate signals consistently.

## Hypothesis 5: Post-Signal Drift Validation

### Objective

Determine whether the generated signals contain predictive information before building a full backtest.

The strategy assumes a lead-lag relationship:


1. Leader moves first

2. Follower temporarily lags

3. Follower eventually catches up


If this assumption is correct, then after a LONG signal is triggered, the follower should exhibit positive future returns.

---

### Research Question

After a LONG signal appears, does the follower generate positive forward returns?

More specifically:

- Does the follower rise over the next 5 minutes?
- Does the follower rise over the next 15 minutes?
- Does the follower rise over the next 30 minutes?
- Does the follower rise over the next 60 minutes?

---

### Methodology

For every `long_signal_start` event:

1. Record the follower price at signal time.
2. Compute the follower forward return over:
   - 5 minutes
   - 15 minutes
   - 30 minutes
   - 60 minutes
3. Aggregate all signal events across the sample.
4. Measure the average forward return at each horizon.

Forward return is defined as:

$ 
ForwardReturn_h =
\frac{P_{t+h}}{P_t} - 1 $ 

where:

- $ P_t $ = follower price at signal time
- $ P_{t+h} $ = follower price h minutes later

---

### Hypotheses

#### Null Hypothesis

The signal contains no predictive information.

$ E[ForwardReturn_h \mid Signal] = 0 $

---

#### Alternative Hypothesis

The signal identifies delayed price discovery and the follower tends to move upward after the signal.

$ E[ForwardReturn_h \mid Signal] > 0 $

---

### Why This Test Matters

This is the first direct alpha validation test.

Previous hypotheses validated:

- existence of lead-lag effects,
- usefulness of correlation ranking,
- market-cap leadership,
- signal frequency.

However, none of them prove profitability.

Only post-signal drift can answer:

> Does the follower actually move in the expected direction after the signal?

A positive and persistent forward-return profile would provide evidence that the strategy captures a genuine catch-up effect and therefore justifies moving to a full backtest.

In [69]:
def add_forward_returns(df, horizons=(5, 15, 30, 60)):
    df = df.sort_values("date").copy()

    for h in horizons:
        df[f"follower_fwd_ret_{h}m"] = (
            df["follower_price"].shift(-h) / df["follower_price"] - 1
        )

    return df

In [70]:
pair_df = add_forward_returns(pair_df)

pair_df.loc[pair_df["long_signal_start"],[
        "date",
        "follower_price",
        "follower_fwd_ret_5m",
        "follower_fwd_ret_15m",
        "follower_fwd_ret_30m",
        "follower_fwd_ret_60m"]].head()

,date,follower_price,follower_fwd_ret_5m,follower_fwd_ret_15m,follower_fwd_ret_30m,follower_fwd_ret_60m
312,2021-06-01 14:42:00,81.641197,0.000192,-0.002403,-0.004057,-0.007521
787,2021-06-03 09:35:00,81.750000,-0.004709,-0.013272,-0.009358,-0.006606
1197,2021-06-04 09:54:00,81.205002,0.000616,0.002401,0.001416,0.002655
1564,2021-06-07 09:30:00,81.264999,0.003261,-0.000677,0.002043,0.000677
1569,2021-06-07 09:35:00,81.529999,-0.003483,-0.004232,0.000123,-0.000343


In [71]:
long_signal_forward_returns = pair_df.loc[pair_df["long_signal_start"],
    [
        "follower_fwd_ret_5m",
        "follower_fwd_ret_15m",
        "follower_fwd_ret_30m",
        "follower_fwd_ret_60m"
    ]
]

long_signal_forward_returns.describe()

,follower_fwd_ret_5m,follower_fwd_ret_15m,follower_fwd_ret_30m,follower_fwd_ret_60m
count,762.000000,7.620000e+02,762.000000,762.000000
mean,-0.000356,-4.434270e-04,-0.000408,-0.001285
std,0.005888,9.860815e-03,0.012041,0.014791
min,-0.028328,-7.869029e-02,-0.088573,-0.096755
25%,-0.003144,-4.882425e-03,-0.006140,-0.009507
50%,-0.000380,-7.748604e-07,-0.000299,-0.000742
75%,0.002695,4.775941e-03,0.006010,0.007305
max,0.027247,7.193804e-02,0.046925,0.058778


In [72]:
def compute_long_signal_forward_returns_for_pair(leader, follower, horizons=(5, 15, 30, 60)):
    df = load_pair_prices(leader, follower)

    df["trading_day"] = df["date"].dt.date

    df["leader_sma15"] = df["leader_price"].rolling(15).mean()
    df["follower_sma15"] = df["follower_price"].rolling(15).mean()

    df["leader_long_sma"] = df["leader_sma15"] * 1.006
    df["follower_long_sma"] = df["follower_sma15"] * 1.002

    df["long_signal"] = (
        (df["leader_price"] > df["leader_long_sma"]) &
        (df["follower_price"] < df["follower_long_sma"])
    )

    prev_long = (
        df.groupby("trading_day")["long_signal"]
        .shift(1)
        .fillna(False)
        .astype(bool)
    )

    df["long_signal_start"] = df["long_signal"].astype(bool) & (~prev_long)

    for h in horizons:
        df[f"follower_fwd_ret_{h}m"] = (
            df["follower_price"].shift(-h) / df["follower_price"] - 1
        )

    events = df.loc[df["long_signal_start"]].copy()

    events["leader"] = leader
    events["follower"] = follower

    return events[
        ["date", "trading_day", "leader", "follower", "follower_price"]
        + [f"follower_fwd_ret_{h}m" for h in horizons]
    ]


h5_events = []

for i, row in unique_top_pairs.iterrows():
    leader = row["leader"]
    follower = row["follower"]

    print(f"{i+1}/{len(unique_top_pairs)} - {leader}->{follower}")

    pair_events = compute_long_signal_forward_returns_for_pair(
        leader,
        follower
    )

    h5_events.append(pair_events)

h5_long_events_df = pd.concat(h5_events, ignore_index=True)

h5_long_events_df[
    [
        "follower_fwd_ret_5m",
        "follower_fwd_ret_15m",
        "follower_fwd_ret_30m",
        "follower_fwd_ret_60m"
    ]
].describe()

1/118 - GOOG->GOOGL
2/118 - TXN->ADI
3/118 - NXPI->MCHP
4/118 - ADI->MCHP
5/118 - TXN->MCHP
6/118 - XEL->EXC
7/118 - TXN->NXPI
8/118 - ADI->NXPI
9/118 - AMAT->ADI
10/118 - MU->AMAT
11/118 - AMAT->NXPI
12/118 - AMAT->MCHP
13/118 - AMAT->TXN
14/118 - AVGO->TXN
15/118 - AVGO->ADI
16/118 - MSFT->ADBE
17/118 - ADBE->CDNS
18/118 - NVDA->AMD
19/118 - ADI->MRVL
20/118 - SNPS->CDNS
21/118 - MRVL->MCHP
22/118 - AMAT->MRVL
23/118 - TXN->MRVL
24/118 - AVGO->MCHP
25/118 - MU->ADI
26/118 - AMGN->GILD
27/118 - MRVL->NXPI
28/118 - AVGO->AMAT
29/118 - MU->NXPI
30/118 - AVGO->NXPI
31/118 - CDNS->ADSK
32/118 - MU->MCHP
33/118 - MU->TXN
34/118 - ASML->AMAT
35/118 - MSFT->CDNS
36/118 - AVGO->MRVL
37/118 - ASML->NXPI
38/118 - ADBE->SNPS
39/118 - ADBE->ADSK
40/118 - MU->MRVL
41/118 - ASML->MCHP
42/118 - INTU->ADBE
43/118 - INTU->CDNS
44/118 - ASML->ADI
45/118 - MSFT->INTU
46/118 - AVGO->MU
47/118 - MSFT->ADSK
48/118 - INTC->TXN
49/118 - AVGO->ASML
50/118 - SNPS->ADSK
51/118 - NVDA->MRVL
52/118 - AMD->MRVL
53

,follower_fwd_ret_5m,follower_fwd_ret_15m,follower_fwd_ret_30m,follower_fwd_ret_60m
count,85211.000000,85207.000000,85207.000000,85206.000000
mean,0.000034,0.000027,0.000024,0.000009
std,0.005045,0.007580,0.009557,0.011955
min,-0.279402,-0.294136,-0.275213,-0.291781
25%,-0.001921,-0.003191,-0.004224,-0.005645
50%,0.000081,0.000084,0.000128,0.000215
75%,0.002132,0.003473,0.004563,0.005846
max,0.058240,0.076908,0.150021,0.143132


--- 

## Hypothesis 5B: Signal Alpha vs Random Baseline

### Objective

Determine whether the positive post-signal drift observed in H5 is genuinely caused by the signal or simply reflects the normal drift of the universe.

---

### Research Question

Do signal-triggered observations outperform randomly selected observations?

---

### Methodology

Compare:

- Forward returns after a `long_signal_start`
- Forward returns from randomly selected timestamps

using the same horizons:

- 5 minutes
- 15 minutes
- 30 minutes
- 60 minutes

---

### Hypotheses

#### Null Hypothesis

The signal provides no incremental information.

$ E[ForwardReturn_h \mid Signal] = E[ForwardReturn_h \mid Random]$

#### Alternative Hypothesis

The signal identifies superior opportunities.

$ E[ForwardReturn_h \mid Signal] > E[ForwardReturn_h \mid Random] $

---

### Why This Test Matters

A positive forward return alone is not sufficient evidence of alpha.

The signal only has value if it produces returns that are systematically better than the baseline behavior of the universe.

This is the final validation step before proceeding to a full strategy backtest.

In [76]:
# 85,212 Signal Events vs 85,212 Random Non-Signal Events

# forward returns for any timestamp

def build_pair_forward_returns(leader,follower,horizons=(5, 15, 30, 60)):
    
    df = load_pair_prices(leader, follower)
    df["trading_day"] = df["date"].dt.date
    # Signal construction
    df["leader_sma15"] = df["leader_price"].rolling(15).mean()
    df["follower_sma15"] = df["follower_price"].rolling(15).mean()
    df["leader_long_sma"] = df["leader_sma15"] * 1.006
    df["follower_long_sma"] = df["follower_sma15"] * 1.002
    df["long_signal"] = ((df["leader_price"] > df["leader_long_sma"]) & (df["follower_price"] < df["follower_long_sma"]))

    prev_long = (df.groupby("trading_day")["long_signal"].shift(1).fillna(False).astype(bool))

    df["long_signal_start"] = (df["long_signal"].astype(bool) & (~prev_long))

    # Forward returns
    for h in horizons:
        df[f"follower_fwd_ret_{h}m"] = (df["follower_price"].shift(-h) / df["follower_price"] - 1)

    return df

In [77]:
test_df = build_pair_forward_returns("NVDA", "AMD")
test_df["long_signal_start"].sum()

np.int64(762)

In [78]:
def sample_random_non_signal_events_for_pair(leader,follower,n_samples,horizons=(5, 15, 30, 60),random_state=42):
    df = build_pair_forward_returns(leader, follower, horizons=horizons)

    fwd_cols = [f"follower_fwd_ret_{h}m" for h in horizons]

    eligible = df.loc[(~df["long_signal_start"]) & df[fwd_cols].notna().all(axis=1)].copy()

    n = min(n_samples, len(eligible))

    sample = eligible.sample(
        n=n,
        replace=False,
        random_state=random_state
    )

    sample["leader"] = leader
    sample["follower"] = follower

    return sample[
        ["date", "trading_day", "leader", "follower", "follower_price"] + fwd_cols
    ]

In [79]:
random_test = sample_random_non_signal_events_for_pair(
    "NVDA",
    "AMD",
    n_samples=762
)

random_test[
    [
        "follower_fwd_ret_5m",
        "follower_fwd_ret_15m",
        "follower_fwd_ret_30m",
        "follower_fwd_ret_60m"
    ]
].describe()

,follower_fwd_ret_5m,follower_fwd_ret_15m,follower_fwd_ret_30m,follower_fwd_ret_60m
count,762.000000,762.000000,762.000000,762.000000
mean,0.000086,-0.000148,0.000367,0.000541
std,0.004364,0.006379,0.014757,0.015328
min,-0.043364,-0.045570,-0.044302,-0.065356
25%,-0.001452,-0.002682,-0.003807,-0.005072
50%,0.000061,-0.000008,-0.000235,0.000117
75%,0.001506,0.002292,0.003082,0.004995
max,0.066078,0.073395,0.333960,0.266137


In [80]:
random_events = []

for i, row in unique_top_pairs.iterrows():

    leader = row["leader"]
    follower = row["follower"]

    n_pair_events = len(
        h5_long_events_df[
            (h5_long_events_df["leader"] == leader)
            &
            (h5_long_events_df["follower"] == follower)
        ]
    )

    if n_pair_events == 0:
        continue

    print(
        f"{i+1}/{len(unique_top_pairs)} "
        f"{leader}->{follower} "
        f"random={n_pair_events}"
    )

    sample = sample_random_non_signal_events_for_pair(
        leader,
        follower,
        n_samples=n_pair_events
    )

    random_events.append(sample)

random_baseline_df = pd.concat(
    random_events,
    ignore_index=True
)

len(random_baseline_df)

1/118 GOOG->GOOGL random=1
2/118 TXN->ADI random=109
3/118 NXPI->MCHP random=284
4/118 ADI->MCHP random=151
5/118 TXN->MCHP random=95
6/118 XEL->EXC random=130
7/118 TXN->NXPI random=116
8/118 ADI->NXPI random=169
9/118 AMAT->ADI random=634
10/118 MU->AMAT random=954
11/118 AMAT->NXPI random=536
12/118 AMAT->MCHP random=492
13/118 AMAT->TXN random=785
14/118 AVGO->TXN random=939
15/118 AVGO->ADI random=817
16/118 MSFT->ADBE random=178
17/118 ADBE->CDNS random=447
18/118 NVDA->AMD random=762
19/118 ADI->MRVL random=261
20/118 SNPS->CDNS random=283
21/118 MRVL->MCHP random=1312
22/118 AMAT->MRVL random=492
23/118 TXN->MRVL random=213
24/118 AVGO->MCHP random=764
25/118 MU->ADI random=1311
26/118 AMGN->GILD random=260
27/118 MRVL->NXPI random=1383
28/118 AVGO->AMAT random=625
29/118 MU->NXPI random=1126
30/118 AVGO->NXPI random=771
31/118 CDNS->ADSK random=408
32/118 MU->MCHP random=1059
33/118 MU->TXN random=1431
34/118 ASML->AMAT random=257
35/118 MSFT->CDNS random=201
36/118 AVGO->MRVL

85212

In [81]:
signal_means = (h5_long_events_df[["follower_fwd_ret_5m","follower_fwd_ret_15m","follower_fwd_ret_30m","follower_fwd_ret_60m"]].mean())
signal_means

follower_fwd_ret_5m     0.000034
follower_fwd_ret_15m    0.000027
follower_fwd_ret_30m    0.000024
follower_fwd_ret_60m    0.000009
dtype: float32

In [82]:
random_means = (random_baseline_df[["follower_fwd_ret_5m","follower_fwd_ret_15m","follower_fwd_ret_30m","follower_fwd_ret_60m"]].mean())
random_means

follower_fwd_ret_5m     0.000013
follower_fwd_ret_15m    0.000030
follower_fwd_ret_30m    0.000084
follower_fwd_ret_60m    0.000155
dtype: float32

In [83]:
alpha_comparison = pd.DataFrame({
    "signal": signal_means,
    "random": random_means
})

alpha_comparison["excess_return"] = (
    alpha_comparison["signal"]
    - alpha_comparison["random"]
)

alpha_comparison

,signal,random,excess_return
follower_fwd_ret_5m,0.000034,0.000013,0.000021
follower_fwd_ret_15m,0.000027,0.000030,-0.000003
follower_fwd_ret_30m,0.000024,0.000084,-0.000060
follower_fwd_ret_60m,0.000009,0.000155,-0.000147


--- 
#  p2


In [84]:
# crear llave pair para eventos
h5_check = h5_long_events_df.copy()
h5_check["pair_key"] = (h5_check["leader"] + "_" + h5_check["follower"]
)

# crear llave pair para Top50 diario
top50_check = daily_top50_pairs.copy()
top50_check["pair_key"] = (
    top50_check["leader"] + "_" + top50_check["follower"]
)

top50_daily_keys = top50_check[
    ["trading_day", "pair_key"]
].drop_duplicates()

# cruzar eventos con top50 del mismo día
h5_check = h5_check.merge(
    top50_daily_keys.assign(in_top50_that_day=True),
    on=["trading_day", "pair_key"],
    how="left"
)

h5_check["in_top50_that_day"] = (
    h5_check["in_top50_that_day"].fillna(False)
)

h5_check["in_top50_that_day"].value_counts(normalize=True)

in_top50_that_day
False    0.601359
True     0.398641
Name: proportion, dtype: float64

In [85]:
h5_check["in_top50_that_day"].value_counts()

in_top50_that_day
False    51243
True     33969
Name: count, dtype: int64

In [86]:
h5_long_events_top50_only = h5_check[
    h5_check["in_top50_that_day"]
].copy()

len(h5_long_events_df), len(h5_long_events_top50_only)

(85212, 33969)

In [87]:
signal_means_top50 = (
    h5_long_events_top50_only[
        [
            "follower_fwd_ret_5m",
            "follower_fwd_ret_15m",
            "follower_fwd_ret_30m",
            "follower_fwd_ret_60m"
        ]
    ]
    .mean()
)

signal_means_top50

follower_fwd_ret_5m     0.000089
follower_fwd_ret_15m    0.000090
follower_fwd_ret_30m    0.000156
follower_fwd_ret_60m    0.000119
dtype: float32

In [ ]:
random_check = random_baseline_df.copy()

random_check["pair_key"] = (
    random_check["leader"] + "_" + random_check["follower"]
)

random_check = random_check.merge(
    top50_daily_keys.assign(in_top50_that_day=True),
    on=["trading_day", "pair_key"],
    how="left"
)

random_check["in_top50_that_day"] = (
    random_check["in_top50_that_day"].fillna(False)
)

random_check["in_top50_that_day"].value_counts(normalize=True)

in_top50_that_day
False    0.58304
True     0.41696
Name: proportion, dtype: float64

In [90]:
random_baseline_top50_only = random_check[random_check["in_top50_that_day"]
].copy()

len(random_baseline_df), len(random_baseline_top50_only)

(85212, 35530)

In [91]:
random_means_top50 = (random_baseline_top50_only[[
            "follower_fwd_ret_5m",
            "follower_fwd_ret_15m",
            "follower_fwd_ret_30m",
            "follower_fwd_ret_60m"]].mean())
random_means_top50

follower_fwd_ret_5m     0.000004
follower_fwd_ret_15m    0.000020
follower_fwd_ret_30m    0.000075
follower_fwd_ret_60m    0.000139
dtype: float32

In [92]:
alpha_comparison_top50 = pd.DataFrame({
    "signal_top50": signal_means_top50,
    "random_top50": random_means_top50})

alpha_comparison_top50["excess_return"] = (
    alpha_comparison_top50["signal_top50"]
    - alpha_comparison_top50["random_top50"]
)

alpha_comparison_top50

,signal_top50,random_top50,excess_return
follower_fwd_ret_5m,0.000089,0.000004,0.000085
follower_fwd_ret_15m,0.000090,0.000020,0.000070
follower_fwd_ret_30m,0.000156,0.000075,0.000081
follower_fwd_ret_60m,0.000119,0.000139,-0.000020


In [93]:
from scipy.stats import ttest_ind

ttest_ind(
    h5_long_events_top50_only["follower_fwd_ret_5m"],
    random_baseline_top50_only["follower_fwd_ret_5m"],
    equal_var=False
)

TtestResult(statistic=np.float32(2.694181), pvalue=np.float32(0.007058286), df=np.float32(56031.59))

There is evidence of a real signal, but its economic magnitude appears small and requires validation through a full backtest with real entry and exit rules.